# Avaliação oficial das tabelas Oracle

Notebook de estudo técnico das tabelas envolvidas na rotina de alimentação Oracle.

## Segurança

- Usa o cliente Oracle existente em `src` e o ambiente `desenv.env`.
- Aceita execução apenas em `MODELAGEM`.
- É **somente leitura**: não contém `INSERT`, `UPDATE`, `DELETE`, `TRUNCATE`, `DROP` ou chamadas de recarga.
- Cada consulta é isolada por `try/except`; uma permissão ausente ou uma tabela com problema não impede o estudo das demais.
- As mensagens usam `[OK]`, `[INFO]`, `[AVISO]` e `[ERRO]` para facilitar a leitura.

## Escopo

`PMPT_TCN`, `PBCO_CADD`, `SGT_CADD`, `TND_CGTV_TCN`, `APSC_TCN`, `CNR_TCN`, `AVS_SELD`, `PROJ_CADD`, `RCM_VLDD` e `RCM_VRS`.

Ao final, a variável remota `documentacao_tabelas_envolvidas_md` contém um relatório Markdown construído com os resultados reais desta execução.


## 1. Criar a sessão Spark local

Este início mantém o padrão oficial do projeto. Para outro destino, a única configuração prevista é `nome_arquivo_env_modelagem`; nesta avaliação ela permanece como `desenv.env`.


In [ ]:
from src.utils.gerenciador_sessao_spark_local import (
    GerenciadorSessaoSpark,
    ler_variavel_ambiente_local,
)

spark = None
gerenciador_spark = None

try:
    ambiente = ler_variavel_ambiente_local("AMBIENTE").strip().upper()
    print(f"[INFO] Ambiente solicitado: {ambiente}")

    if ambiente != "MODELAGEM":
        raise RuntimeError(
            "Avaliação bloqueada: este notebook deve ser executado somente em MODELAGEM."
        )

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="avaliacao_oficial_tabelas_oracle",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=["/dados/shared/bin/ojdbc8.jar"],
        spark_conf={"spark.driver.memoryOverhead": "8g"},
    )
    print("[OK] Sessão Spark criada com as configurações de desenvolvimento.")
except Exception as exc:
    print(f"[ERRO] Não foi possível criar a sessão Spark: {type(exc).__name__}: {exc}")
    print("[AVISO] Corrija a conexão e execute novamente esta célula antes de continuar.")


## 2. Carregar o cliente Oracle do `src`

A célula seguinte disponibiliza `criar_cliente_oracle_spark` na sessão remota. Se a sessão anterior não estiver ativa, ela exibirá o erro sem modificar o Oracle.


In [ ]:
try:
    if spark is None:
        raise RuntimeError("A sessão Spark local não foi criada.")
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
    print("[OK] Utilitários remotos do src carregados.")
except Exception as exc:
    print(f"[ERRO] Falha ao carregar os utilitários remotos: {type(exc).__name__}: {exc}")


## 3. Configuração da avaliação e conexão Oracle

Toda a análise a partir daqui ocorre na sessão Spark remota e começa com `%%spark`.


In [ ]:
%%spark

from collections import defaultdict, deque
from datetime import datetime
import json
import re
import traceback

TABELAS_ENVOLVIDAS = [
    "PMPT_TCN",
    "PBCO_CADD",
    "SGT_CADD",
    "TND_CGTV_TCN",
    "APSC_TCN",
    "CNR_TCN",
    "AVS_SELD",
    "PROJ_CADD",
    "RCM_VLDD",
    "RCM_VRS",
]

estado_avaliacao = {
    "inicio": datetime.now().isoformat(timespec="seconds"),
    "consultas_ok": [],
    "avisos": [],
    "erros": [],
}


def mensagem(nivel, texto):
    nivel = str(nivel).strip().upper()
    print(f"[{nivel}] {texto}")
    if nivel == "AVISO":
        estado_avaliacao["avisos"].append(texto)
    elif nivel == "ERRO":
        estado_avaliacao["erros"].append(texto)


def cabecalho(titulo):
    print("\n" + "=" * 88)
    print(titulo)
    print("=" * 88)


def sql_lista(valores):
    seguros = []
    for valor in valores:
        texto = str(valor).strip().upper()
        if not re.fullmatch(r"[A-Z][A-Z0-9_#$]{0,127}", texto):
            raise ValueError(f"Identificador Oracle inválido: {valor!r}")
        seguros.append("'" + texto + "'")
    return ", ".join(seguros)


cliente_oracle = None
OWNER = None
contexto_banco = {}

try:
    ambiente_remoto = ler_variavel_ambiente_spark("AMBIENTE").strip().upper()
    if ambiente_remoto != "MODELAGEM":
        raise RuntimeError(
            f"Ambiente remoto inesperado: {ambiente_remoto}. Esperado: MODELAGEM."
        )

    cliente_oracle = criar_cliente_oracle_spark()
    OWNER = str(cliente_oracle.schema).strip().upper()

    if not re.fullmatch(r"[A-Z][A-Z0-9_#$]{0,127}", OWNER):
        raise ValueError(f"Schema Oracle inválido: {OWNER!r}")

    linha = cliente_oracle.run_select(
        """
        SELECT
            SYS_CONTEXT('USERENV', 'DB_NAME') AS DB_NAME,
            SYS_CONTEXT('USERENV', 'SERVICE_NAME') AS SERVICE_NAME,
            SYS_CONTEXT('USERENV', 'CURRENT_SCHEMA') AS CURRENT_SCHEMA,
            SYS_CONTEXT('USERENV', 'SESSION_USER') AS SESSION_USER
        FROM DUAL
        """
    ).collect()[0].asDict(recursive=True)
    contexto_banco = {str(k).upper(): v for k, v in linha.items()}

    cabecalho("CONEXÃO ORACLE")
    mensagem("OK", f"Cliente Oracle criado para o schema {OWNER}.")
    mensagem("INFO", f"Banco: {contexto_banco.get('DB_NAME')}")
    mensagem("INFO", f"Serviço: {contexto_banco.get('SERVICE_NAME')}")
    mensagem("INFO", f"Usuário da sessão: {contexto_banco.get('SESSION_USER')}")
    mensagem("INFO", "Modo de avaliação: SOMENTE LEITURA.")
except Exception as exc:
    mensagem("ERRO", f"Falha na conexão Oracle: {type(exc).__name__}: {exc}")
    mensagem("AVISO", "As próximas células continuarão, mas consultas sem conexão serão ignoradas.")


## 4. Contrato técnico esperado

Este contrato serve como régua para comparar o DDL conhecido com o dicionário do Oracle. Ele não altera o banco. Colunas extras, ausentes, tipos, nulidade, tamanho e identidade serão conferidos.


In [ ]:
%%spark

CONTRATO_ESPERADO = {}
CHAVES_PRIMARIAS_ESPERADAS = {}
IDS_RELEVANTES = {}


def coluna(tipo, nullable, tamanho=None, precisao=None, escala=None, identity=False):
    return {
        "tipo": tipo,
        "nullable": nullable,
        "tamanho": tamanho,
        "precisao": precisao,
        "escala": escala,
        "identity": identity,
    }


try:
    CONTRATO_ESPERADO = {
        "PMPT_TCN": {
            "NR_PMPT_IDFR": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_PMPT_PDRO": coluna("VARCHAR2", False, tamanho=100),
            "NR_ETP_PSCL": coluna("NUMBER", False, precisao=3, escala=0),
            "NR_VRS_ATU": coluna("NUMBER", False, precisao=3, escala=0),
            "TX_FUC_OPRL_PMPT": coluna("CLOB", False),
            "TX_DTZ_ORTR_PMPT": coluna("CLOB", False),
            "TX_PDRO_DFND_PMPT": coluna("CLOB", False),
            "QT_VRV_TTL": coluna("NUMBER", True, precisao=3, escala=0),
        },
        "PBCO_CADD": {
            "NR_IDFR_PBCO": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_PBCO_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_DCR_DETD_PBCO": coluna("CLOB", False),
            "TX_NCDD_RCNL_PBCO": coluna("CLOB", False),
            "TX_NCDD_EMOC_PBCO": coluna("CLOB", False),
            "TX_TRM_OBG_PBCO": coluna("CLOB", True),
            "TX_TRM_N_PMT_PBCO": coluna("CLOB", True),
            "NM_DB": coluna("VARCHAR2", False, tamanho=100),
            "NM_TAB_DB": coluna("VARCHAR2", False, tamanho=100),
            "NM_COL_TAB": coluna("VARCHAR2", True, tamanho=100),
            "TS_CAD": coluna("TIMESTAMP", False),
            "CD_EST_PBCO": coluna("NUMBER", False, precisao=1, escala=0),
            "CD_TIP_TAB": coluna("NUMBER", False, precisao=1, escala=0),
            "CD_USU_RSP_CAD": coluna("VARCHAR2", False, tamanho=8),
        },
        "SGT_CADD": {
            "NR_IDFR_SGT": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "TX_REG_EXNO_SGT": coluna("VARCHAR2", True, tamanho=100),
            "NM_SGT_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_DCR_DETD_SGT": coluna("CLOB", False),
            "TX_BNF_RCNL_SGT": coluna("CLOB", False),
            "TX_BNF_EMOC_SGT": coluna("CLOB", False),
            "TX_TRM_OBG_SGT": coluna("CLOB", True),
            "TX_TRM_N_PMT_SGT": coluna("CLOB", True),
            "TS_CAD_SGT": coluna("TIMESTAMP", False),
            "CD_EST_SGT": coluna("NUMBER", False, precisao=1, escala=0),
            "CD_USU_RSP_CAD_SGT": coluna("VARCHAR2", False, tamanho=8),
        },
        "TND_CGTV_TCN": {
            "NR_IDFR_TND": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_TND_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_DCR_DETD_TND": coluna("CLOB", False),
            "TX_FUC_OPRL_TND": coluna("CLOB", False),
            "TX_RCPO_PDRO_TND": coluna("CLOB", True),
        },
        "APSC_TCN": {
            "NR_IDFR_APSC": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_APSC_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_FUC_OPRL_APSC": coluna("CLOB", False),
            "TX_DCR_DETD_APSC": coluna("CLOB", False),
            "TX_RCPO_PDRO_APSC": coluna("CLOB", True),
        },
        "CNR_TCN": {
            "NR_IDFR_CNR": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_CNR_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_FUC_OPRL_CNR": coluna("CLOB", False),
            "TX_DCR_DETD_CNR": coluna("CLOB", False),
            "TX_RCPO_PDRO_CNR": coluna("CLOB", True),
        },
        "AVS_SELD": {
            "NR_AVS_SELD": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_IDFR_PBCO": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_IDFR_SGT": coluna("NUMBER", False, precisao=11, escala=0),
            "CD_EXNO_CRIC": coluna("VARCHAR2", False, tamanho=36),
            "CD_EXNO_AVS": coluna("VARCHAR2", False, tamanho=36),
            "TX_TIT_PDRO_AVS": coluna("CLOB", False),
            "TX_STIT_PDRO_AVS": coluna("CLOB", False),
            "TX_ACMT_PDRO_AVS": coluna("CLOB", False),
            "JS_AVS_PDRO": coluna("CLOB", False),
            "TS_CAD_AVS": coluna("TIMESTAMP", False),
            "CD_USU_RSP_CAD": coluna("VARCHAR2", False, tamanho=8),
            "NR_IDFR_APSC": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_IDFR_CNR": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_IDFR_TND": coluna("NUMBER", False, precisao=11, escala=0),
        },
        "PROJ_CADD": {
            "NR_IDFR_PROJ": coluna("NUMBER", False, precisao=11, escala=0, identity=True),
            "NM_PROJ_CMT": coluna("VARCHAR2", False, tamanho=500),
            "TX_DCR_DETD": coluna("VARCHAR2", False, tamanho=750),
            "TS_CAD": coluna("TIMESTAMP", False),
            "CD_EST_PROJ": coluna("NUMBER", False, precisao=1, escala=0),
            "CD_USU_RSP_CAD_PROJ": coluna("VARCHAR2", False, tamanho=8),
        },
        "RCM_VLDD": {
            "NR_IDFR_RCM": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_IDFR_PROJ": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_VLDO_VRS": coluna("NUMBER", False, precisao=3, escala=0),
            "NR_VRS_PRPT": coluna("NUMBER", False, precisao=3, escala=0),
            "NR_PRI_SELD": coluna("NUMBER", False, precisao=3, escala=0),
            "TS_ULT_ATL": coluna("TIMESTAMP", False),
            "NM_RCM_PDRO": coluna("VARCHAR2", False, tamanho=500),
            "TX_DCR_DETD": coluna("VARCHAR2", False, tamanho=750),
            "CD_EST_RCM": coluna("NUMBER", False, precisao=1, escala=0),
        },
        "RCM_VRS": {
            "NR_IDFR_RCM": coluna("NUMBER", False, precisao=11, escala=0),
            "NR_VRS_PRPT": coluna("NUMBER", False, precisao=3, escala=0),
            "NR_AVS_SELD": coluna("NUMBER", False, precisao=11, escala=0),
            "TX_DCR_DETD": coluna("VARCHAR2", False, tamanho=750),
            "DT_PRPT_INC_RCM": coluna("DATE", False),
            "DT_PRPT_FIM_RCM": coluna("DATE", False),
            "TX_PRPT_PRM_HDR_API": coluna("CLOB", False),
            "TX_MTV_AVLC": coluna("VARCHAR2", True, tamanho=1000),
            "TS_ULT_ALT": coluna("TIMESTAMP", False),
            "NR_IDFR_PROJ": coluna("NUMBER", False, precisao=11, escala=0),
            "CD_USU_RSP_CAD": coluna("VARCHAR2", False, tamanho=8),
            "CD_EST_RCM_VRS": coluna("NUMBER", False, precisao=1, escala=0),
        },
    }

    CHAVES_PRIMARIAS_ESPERADAS = {
        "PMPT_TCN": ["NR_PMPT_IDFR"],
        "PBCO_CADD": ["NR_IDFR_PBCO"],
        "SGT_CADD": ["NR_IDFR_SGT"],
        "TND_CGTV_TCN": ["NR_IDFR_TND"],
        "APSC_TCN": ["NR_IDFR_APSC"],
        "CNR_TCN": ["NR_IDFR_CNR"],
        "AVS_SELD": ["NR_IDFR_PBCO", "NR_IDFR_SGT", "NR_AVS_SELD"],
        "PROJ_CADD": ["NR_IDFR_PROJ"],
        "RCM_VLDD": ["NR_IDFR_RCM", "NR_IDFR_PROJ"],
        "RCM_VRS": ["NR_IDFR_RCM", "NR_IDFR_PROJ", "NR_VRS_PRPT"],
    }

    IDS_RELEVANTES = {
        "PMPT_TCN": ["NR_PMPT_IDFR"],
        "PBCO_CADD": ["NR_IDFR_PBCO"],
        "SGT_CADD": ["NR_IDFR_SGT"],
        "TND_CGTV_TCN": ["NR_IDFR_TND"],
        "APSC_TCN": ["NR_IDFR_APSC"],
        "CNR_TCN": ["NR_IDFR_CNR"],
        "AVS_SELD": [
            "NR_AVS_SELD",
            "NR_IDFR_PBCO",
            "NR_IDFR_SGT",
            "NR_IDFR_APSC",
            "NR_IDFR_CNR",
            "NR_IDFR_TND",
        ],
        "PROJ_CADD": ["NR_IDFR_PROJ"],
        "RCM_VLDD": ["NR_IDFR_RCM", "NR_IDFR_PROJ"],
        "RCM_VRS": ["NR_IDFR_RCM", "NR_IDFR_PROJ", "NR_VRS_PRPT", "NR_AVS_SELD"],
    }

    mensagem("OK", f"Contrato esperado definido para {len(CONTRATO_ESPERADO)} tabelas.")
except Exception as exc:
    mensagem("ERRO", f"Falha ao definir o contrato esperado: {type(exc).__name__}: {exc}")


## 5. Funções de consulta resiliente

Todas as consultas passam por um único ponto de tratamento. O SQL não é exibido em erros para evitar ruído e exposição acidental; ficam registrados o rótulo, o tipo e a mensagem da falha.


In [ ]:
%%spark

resultados_consultas = {}


def normalizar_linha(linha):
    bruto = linha.asDict(recursive=True) if hasattr(linha, "asDict") else dict(linha)
    return {str(chave).strip().upper(): valor for chave, valor in bruto.items()}


def consultar_seguro(rotulo, sql, mostrar_sucesso=True):
    if cliente_oracle is None:
        texto = f"{rotulo}: consulta ignorada porque o cliente Oracle não está disponível."
        mensagem("AVISO", texto)
        resultados_consultas[rotulo] = []
        return []

    try:
        linhas_coletadas = cliente_oracle.run_select(sql).collect()
        dados = [normalizar_linha(linha) for linha in linhas_coletadas]
        resultados_consultas[rotulo] = dados
        estado_avaliacao["consultas_ok"].append(rotulo)
        if mostrar_sucesso:
            mensagem("OK", f"{rotulo}: {len(dados)} linha(s) retornada(s).")
        return dados
    except Exception as exc:
        texto = f"{rotulo}: {type(exc).__name__}: {exc}"
        mensagem("ERRO", texto)
        resultados_consultas[rotulo] = []
        return []


def valor_inteiro(valor, padrao=None):
    try:
        return int(valor) if valor is not None else padrao
    except (TypeError, ValueError):
        return padrao


def por_tabela(linhas):
    saida = defaultdict(list)
    for item in linhas:
        saida[str(item.get("TABLE_NAME", "")).upper()].append(item)
    return dict(saida)


def mostrar_linhas(titulo, linhas, colunas=None, limite=100):
    cabecalho(titulo)
    if not linhas:
        mensagem("INFO", "Nenhuma linha disponível para exibição.")
        return
    dados = linhas[:limite]
    if colunas:
        dados = [{coluna: linha.get(coluna) for coluna in colunas} for linha in dados]
    try:
        spark.createDataFrame(dados).show(n=len(dados), truncate=False)
    except Exception as exc:
        mensagem("AVISO", f"Não foi possível criar a visualização Spark: {exc}")
        for linha in dados:
            print(linha)
    if len(linhas) > limite:
        mensagem("AVISO", f"Exibição limitada a {limite} de {len(linhas)} linhas.")


mensagem("OK", "Funções resilientes de consulta e exibição preparadas.")


## 6. Inventário do dicionário Oracle

Coleta existência, status, organização, estatísticas, colunas, identidades, constraints, chaves estrangeiras, índices, triggers, comentários e privilégios. Cada visão é consultada separadamente.


In [ ]:
%%spark

objetos = []
tabelas_meta = []
colunas_meta = []
identidades = []
constraints = []
checks = []
fks = []
indices = []
triggers = []
comentarios_tabela = []
comentarios_coluna = []
privilegios = []
dependencias_externas = []

try:
    lista_tabelas_sql = sql_lista(TABELAS_ENVOLVIDAS)
    cabecalho("COLETA DO DICIONÁRIO ORACLE")

    objetos = consultar_seguro(
        "Objetos e status",
        f"""
        SELECT OBJECT_NAME AS TABLE_NAME, OBJECT_TYPE, STATUS, CREATED, LAST_DDL_TIME
        FROM ALL_OBJECTS
        WHERE OWNER = '{OWNER}'
          AND OBJECT_NAME IN ({lista_tabelas_sql})
        ORDER BY OBJECT_NAME, OBJECT_TYPE
        """,
    ) if OWNER else []

    tabelas_meta = consultar_seguro(
        "Metadados das tabelas",
        f"""
        SELECT TABLE_NAME, TABLESPACE_NAME, STATUS, NUM_ROWS, BLOCKS,
               LAST_ANALYZED, TEMPORARY, PARTITIONED, IOT_TYPE
        FROM ALL_TABLES
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME
        """,
    ) if OWNER else []

    colunas_meta = consultar_seguro(
        "Colunas físicas",
        f"""
        SELECT TABLE_NAME, COLUMN_ID, COLUMN_NAME, DATA_TYPE, DATA_LENGTH,
               CHAR_LENGTH, CHAR_USED, DATA_PRECISION, DATA_SCALE, NULLABLE
        FROM ALL_TAB_COLUMNS
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME, COLUMN_ID
        """,
    ) if OWNER else []

    identidades = consultar_seguro(
        "Colunas identity",
        f"""
        SELECT TABLE_NAME, COLUMN_NAME, GENERATION_TYPE, SEQUENCE_NAME, IDENTITY_OPTIONS
        FROM ALL_TAB_IDENTITY_COLS
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME, COLUMN_NAME
        """,
    ) if OWNER else []

    constraints = consultar_seguro(
        "Constraints",
        f"""
        SELECT c.TABLE_NAME, c.CONSTRAINT_NAME, c.CONSTRAINT_TYPE, c.STATUS,
               c.VALIDATED, c.DEFERRABLE, c.DEFERRED, c.DELETE_RULE,
               c.R_OWNER, c.R_CONSTRAINT_NAME,
               cc.COLUMN_NAME, cc.POSITION
        FROM ALL_CONSTRAINTS c
        LEFT JOIN ALL_CONS_COLUMNS cc
          ON cc.OWNER = c.OWNER
         AND cc.CONSTRAINT_NAME = c.CONSTRAINT_NAME
         AND cc.TABLE_NAME = c.TABLE_NAME
        WHERE c.OWNER = '{OWNER}'
          AND c.TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY c.TABLE_NAME, c.CONSTRAINT_NAME, cc.POSITION
        """,
    ) if OWNER else []

    checks = consultar_seguro(
        "Condições CHECK",
        f"""
        SELECT TABLE_NAME, CONSTRAINT_NAME, STATUS, VALIDATED, SEARCH_CONDITION_VC
        FROM ALL_CONSTRAINTS
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
          AND CONSTRAINT_TYPE = 'C'
        ORDER BY TABLE_NAME, CONSTRAINT_NAME
        """,
    ) if OWNER else []

    fks = consultar_seguro(
        "Relacionamentos FK",
        f"""
        SELECT c.TABLE_NAME, c.CONSTRAINT_NAME, cc.COLUMN_NAME, cc.POSITION,
               c.STATUS, c.VALIDATED, c.DELETE_RULE,
               rc.OWNER AS REFERENCED_OWNER,
               rc.TABLE_NAME AS REFERENCED_TABLE,
               rcc.COLUMN_NAME AS REFERENCED_COLUMN
        FROM ALL_CONSTRAINTS c
        JOIN ALL_CONS_COLUMNS cc
          ON cc.OWNER = c.OWNER
         AND cc.CONSTRAINT_NAME = c.CONSTRAINT_NAME
         AND cc.TABLE_NAME = c.TABLE_NAME
        JOIN ALL_CONSTRAINTS rc
          ON rc.OWNER = c.R_OWNER
         AND rc.CONSTRAINT_NAME = c.R_CONSTRAINT_NAME
        JOIN ALL_CONS_COLUMNS rcc
          ON rcc.OWNER = rc.OWNER
         AND rcc.CONSTRAINT_NAME = rc.CONSTRAINT_NAME
         AND rcc.POSITION = cc.POSITION
        WHERE c.OWNER = '{OWNER}'
          AND c.TABLE_NAME IN ({lista_tabelas_sql})
          AND c.CONSTRAINT_TYPE = 'R'
        ORDER BY c.TABLE_NAME, c.CONSTRAINT_NAME, cc.POSITION
        """,
    ) if OWNER else []

    indices = consultar_seguro(
        "Índices e colunas",
        f"""
        SELECT i.TABLE_NAME, i.INDEX_NAME, i.INDEX_TYPE, i.UNIQUENESS,
               i.STATUS, i.VISIBILITY, ic.COLUMN_NAME, ic.COLUMN_POSITION
        FROM ALL_INDEXES i
        LEFT JOIN ALL_IND_COLUMNS ic
          ON ic.INDEX_OWNER = i.OWNER
         AND ic.INDEX_NAME = i.INDEX_NAME
         AND ic.TABLE_OWNER = i.TABLE_OWNER
         AND ic.TABLE_NAME = i.TABLE_NAME
        WHERE i.TABLE_OWNER = '{OWNER}'
          AND i.TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY i.TABLE_NAME, i.INDEX_NAME, ic.COLUMN_POSITION
        """,
    ) if OWNER else []

    triggers = consultar_seguro(
        "Triggers",
        f"""
        SELECT TABLE_NAME, TRIGGER_NAME, STATUS, TRIGGER_TYPE, TRIGGERING_EVENT
        FROM ALL_TRIGGERS
        WHERE TABLE_OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME, TRIGGER_NAME
        """,
    ) if OWNER else []

    comentarios_tabela = consultar_seguro(
        "Comentários de tabelas",
        f"""
        SELECT TABLE_NAME, COMMENTS
        FROM ALL_TAB_COMMENTS
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME
        """,
    ) if OWNER else []

    comentarios_coluna = consultar_seguro(
        "Comentários de colunas",
        f"""
        SELECT TABLE_NAME, COLUMN_NAME, COMMENTS
        FROM ALL_COL_COMMENTS
        WHERE OWNER = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME, COLUMN_NAME
        """,
    ) if OWNER else []

    privilegios = consultar_seguro(
        "Privilégios concedidos",
        f"""
        SELECT TABLE_NAME, GRANTEE, PRIVILEGE, GRANTABLE, GRANTOR
        FROM ALL_TAB_PRIVS
        WHERE TABLE_SCHEMA = '{OWNER}'
          AND TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY TABLE_NAME, GRANTEE, PRIVILEGE
        """,
    ) if OWNER else []

    dependencias_externas = consultar_seguro(
        "FKs que apontam para as tabelas estudadas",
        f"""
        SELECT c.OWNER AS CHILD_OWNER, c.TABLE_NAME AS CHILD_TABLE,
               c.CONSTRAINT_NAME, rc.TABLE_NAME AS PARENT_TABLE,
               c.STATUS, c.DELETE_RULE
        FROM ALL_CONSTRAINTS c
        JOIN ALL_CONSTRAINTS rc
          ON rc.OWNER = c.R_OWNER
         AND rc.CONSTRAINT_NAME = c.R_CONSTRAINT_NAME
        WHERE c.CONSTRAINT_TYPE = 'R'
          AND rc.OWNER = '{OWNER}'
          AND rc.TABLE_NAME IN ({lista_tabelas_sql})
        ORDER BY rc.TABLE_NAME, c.OWNER, c.TABLE_NAME
        """,
    ) if OWNER else []
except Exception as exc:
    mensagem("ERRO", f"Falha geral no inventário: {type(exc).__name__}: {exc}")

mostrar_linhas(
    "RESUMO DOS OBJETOS",
    objetos,
    ["TABLE_NAME", "OBJECT_TYPE", "STATUS", "CREATED", "LAST_DDL_TIME"],
)


## 7. Comparar schema real com o contrato

Confere existência das dez tabelas, conjunto exato de colunas, ordem física, tipo, tamanho, precisão, escala, nulidade e presença de identity.


In [ ]:
%%spark

validacao_schema = []
divergencias_schema = []

try:
    colunas_por_tabela = por_tabela(colunas_meta)
    identidades_chave = {
        (str(item.get("TABLE_NAME", "")).upper(), str(item.get("COLUMN_NAME", "")).upper()): item
        for item in identidades
    }
    metadados_identity_disponiveis = (
        "Colunas identity" in estado_avaliacao["consultas_ok"]
    )
    tabelas_encontradas = {
        str(item.get("TABLE_NAME", "")).upper()
        for item in tabelas_meta
    }

    cabecalho("VALIDAÇÃO DO SCHEMA")
    for tabela in TABELAS_ENVOLVIDAS:
        esperado = CONTRATO_ESPERADO.get(tabela, {})
        reais_lista = colunas_por_tabela.get(tabela, [])
        reais = {
            str(item.get("COLUMN_NAME", "")).upper(): item
            for item in reais_lista
        }

        problemas = []
        incompletudes = []
        if tabela not in tabelas_encontradas:
            problemas.append("tabela não localizada em ALL_TABLES")

        ausentes = sorted(set(esperado) - set(reais))
        extras = sorted(set(reais) - set(esperado))
        if ausentes:
            problemas.append("colunas ausentes: " + ", ".join(ausentes))
        if extras:
            problemas.append("colunas extras: " + ", ".join(extras))

        for nome_coluna in sorted(set(esperado) & set(reais)):
            regra = esperado[nome_coluna]
            atual = reais[nome_coluna]
            tipo_atual = str(atual.get("DATA_TYPE") or "").upper()
            tipo_ok = (
                tipo_atual == regra["tipo"]
                or (regra["tipo"] == "TIMESTAMP" and tipo_atual.startswith("TIMESTAMP"))
            )
            if not tipo_ok:
                problemas.append(
                    f"{nome_coluna}: tipo {tipo_atual!r}, esperado {regra['tipo']!r}"
                )

            nullable_atual = str(atual.get("NULLABLE") or "").upper() == "Y"
            if nullable_atual != regra["nullable"]:
                problemas.append(
                    f"{nome_coluna}: nullable={nullable_atual}, esperado={regra['nullable']}"
                )

            if regra["tipo"] == "VARCHAR2" and regra["tamanho"] is not None:
                tamanho_atual = valor_inteiro(atual.get("CHAR_LENGTH"))
                if tamanho_atual != regra["tamanho"]:
                    problemas.append(
                        f"{nome_coluna}: tamanho={tamanho_atual}, esperado={regra['tamanho']}"
                    )

            if regra["tipo"] == "NUMBER":
                precisao_atual = valor_inteiro(atual.get("DATA_PRECISION"))
                escala_atual = valor_inteiro(atual.get("DATA_SCALE"))
                if regra["precisao"] is not None and precisao_atual != regra["precisao"]:
                    problemas.append(
                        f"{nome_coluna}: precisão={precisao_atual}, esperada={regra['precisao']}"
                    )
                if regra["escala"] is not None and escala_atual != regra["escala"]:
                    problemas.append(
                        f"{nome_coluna}: escala={escala_atual}, esperada={regra['escala']}"
                    )

            if metadados_identity_disponiveis:
                identity_atual = (tabela, nome_coluna) in identidades_chave
                if identity_atual != regra["identity"]:
                    problemas.append(
                        f"{nome_coluna}: identity={identity_atual}, esperado={regra['identity']}"
                    )
                elif regra["identity"] and identity_atual:
                    geracao_atual = str(
                        identidades_chave[(tabela, nome_coluna)].get("GENERATION_TYPE") or ""
                    ).upper()
                    if geracao_atual != "ALWAYS":
                        problemas.append(
                            f"{nome_coluna}: geração identity={geracao_atual!r}, esperada='ALWAYS'"
                        )
            elif regra["identity"]:
                incompletudes.append(f"{nome_coluna}: identity não pôde ser avaliada")

        status = (
            "DIVERGENTE" if problemas
            else "PARCIAL" if incompletudes
            else "OK"
        )
        validacao_schema.append({
            "TABELA": tabela,
            "STATUS": status,
            "COLUNAS_ESPERADAS": len(esperado),
            "COLUNAS_ENCONTRADAS": len(reais),
            "DETALHES": " | ".join(problemas + incompletudes) if (problemas or incompletudes) else "Contrato atendido",
        })

        if problemas:
            divergencias_schema.extend([f"{tabela}: {problema}" for problema in problemas])
            mensagem("ERRO", f"{tabela}: {len(problemas)} divergência(s) de schema.")
        elif incompletudes:
            mensagem("AVISO", f"{tabela}: contrato parcialmente avaliado ({len(incompletudes)} item(ns)).")
        else:
            mensagem("OK", f"{tabela}: contrato físico atendido ({len(reais)} colunas).")
except Exception as exc:
    mensagem("ERRO", f"Falha ao comparar schemas: {type(exc).__name__}: {exc}")

mostrar_linhas("RESULTADO DO CONTRATO FÍSICO", validacao_schema)


## 8. Chaves primárias, unicidade e constraints

Além de comparar as PKs reais com o contrato, identifica constraints desabilitadas ou não validadas. A duplicidade dos dados será testada na etapa seguinte.


In [ ]:
%%spark

validacao_pks = []
constraints_problematicas = []
constraints_agrupadas = defaultdict(list)

try:
    if "Constraints" not in estado_avaliacao["consultas_ok"]:
        raise RuntimeError("Metadados de constraints indisponíveis; PKs não serão inferidas.")

    for item in constraints:
        chave = (
            str(item.get("TABLE_NAME", "")).upper(),
            str(item.get("CONSTRAINT_NAME", "")).upper(),
            str(item.get("CONSTRAINT_TYPE", "")).upper(),
        )
        constraints_agrupadas[chave].append(item)

    cabecalho("CHAVES E CONSTRAINTS")
    for tabela in TABELAS_ENVOLVIDAS:
        pks_tabela = []
        for (tab, nome, tipo), itens in constraints_agrupadas.items():
            if tab == tabela and tipo == "P":
                ordenados = sorted(itens, key=lambda x: valor_inteiro(x.get("POSITION"), 9999))
                pks_tabela.append({
                    "NOME": nome,
                    "COLUNAS": [str(x.get("COLUMN_NAME", "")).upper() for x in ordenados],
                    "STATUS": str(ordenados[0].get("STATUS", "")) if ordenados else "",
                    "VALIDATED": str(ordenados[0].get("VALIDATED", "")) if ordenados else "",
                })

        esperada = CHAVES_PRIMARIAS_ESPERADAS.get(tabela, [])
        real = pks_tabela[0]["COLUNAS"] if len(pks_tabela) == 1 else []
        ok = len(pks_tabela) == 1 and real == esperada
        detalhe = (
            f"PK {pks_tabela[0]['NOME']} = {real}" if ok
            else f"esperada={esperada}; encontrada={pks_tabela}"
        )
        validacao_pks.append({"TABELA": tabela, "STATUS": "OK" if ok else "DIVERGENTE", "DETALHES": detalhe})
        mensagem("OK" if ok else "ERRO", f"{tabela}: {detalhe}")

    constraints_vistas = set()
    for item in constraints:
        chave = (item.get("TABLE_NAME"), item.get("CONSTRAINT_NAME"))
        if chave in constraints_vistas:
            continue
        constraints_vistas.add(chave)
        status = str(item.get("STATUS") or "").upper()
        validada = str(item.get("VALIDATED") or "").upper()
        if status != "ENABLED" or validada not in {"VALIDATED", ""}:
            constraints_problematicas.append({
                "TABLE_NAME": item.get("TABLE_NAME"),
                "CONSTRAINT_NAME": item.get("CONSTRAINT_NAME"),
                "CONSTRAINT_TYPE": item.get("CONSTRAINT_TYPE"),
                "STATUS": status,
                "VALIDATED": validada,
            })

    if constraints_problematicas:
        mensagem("AVISO", f"Encontradas {len(constraints_problematicas)} constraints desabilitadas/não validadas.")
    else:
        mensagem("OK", "Nenhuma constraint desabilitada ou não validada foi encontrada.")
except Exception as exc:
    mensagem("ERRO", f"Falha ao avaliar PKs e constraints: {type(exc).__name__}: {exc}")

mostrar_linhas("VALIDAÇÃO DAS CHAVES PRIMÁRIAS", validacao_pks)
mostrar_linhas("CONSTRAINTS QUE EXIGEM ATENÇÃO", constraints_problematicas)


## 9. IDs automáticos e possibilidade de informar ID manualmente

Regra Oracle usada na conclusão:

- `GENERATED ALWAYS`: o Oracle gera o valor e um valor manual explícito é rejeitado (normalmente `ORA-32795`).
- `GENERATED BY DEFAULT` ou `BY DEFAULT ON NULL`: o ID pode ser informado manualmente.
- Sem identity: o Oracle não gera automaticamente por identity; o valor normalmente precisa ser informado, salvo se houver default ou trigger.

Nenhum `INSERT` é executado para chegar a essa conclusão.


In [ ]:
%%spark

relatorio_ids = []

try:
    if "Colunas identity" not in estado_avaliacao["consultas_ok"]:
        raise RuntimeError("Metadados de identity indisponíveis; política de IDs não será inferida.")

    identidade_por_coluna = {
        (str(x.get("TABLE_NAME", "")).upper(), str(x.get("COLUMN_NAME", "")).upper()): x
        for x in identidades
    }
    triggers_por_tabela = por_tabela(triggers)

    cabecalho("POLÍTICA DOS IDs")
    for tabela in TABELAS_ENVOLVIDAS:
        for nome_coluna in IDS_RELEVANTES.get(tabela, []):
            identidade = identidade_por_coluna.get((tabela, nome_coluna))
            geracao = str(identidade.get("GENERATION_TYPE") or "").upper() if identidade else ""

            if identidade and geracao == "ALWAYS":
                automatico = "SIM"
                manual = "NÃO"
                conclusao = "ID gerado automaticamente; valor manual explícito causa erro Oracle (ORA-32795)."
            elif identidade and "BY DEFAULT" in geracao:
                automatico = "SIM"
                manual = "SIM"
                conclusao = "Identity BY DEFAULT: Oracle pode gerar, mas valor manual é aceito."
            elif identidade:
                automatico = "SIM"
                manual = "INDETERMINADO"
                conclusao = f"Identity com modalidade não reconhecida: {geracao}."
            else:
                automatico = "NÃO POR IDENTITY"
                manual = "SIM/OBRIGATÓRIO"
                conclusao = "Sem identity: o valor deve ser tratado pela aplicação ou por outra regra do banco."
                if triggers_por_tabela.get(tabela):
                    conclusao += " Há trigger na tabela; revisar sua lógica antes de concluir."

            item = {
                "TABELA": tabela,
                "COLUNA": nome_coluna,
                "IDENTITY": "SIM" if identidade else "NÃO",
                "GENERATION_TYPE": geracao or "-",
                "SEQUENCE_NAME": identidade.get("SEQUENCE_NAME") if identidade else None,
                "GERADO_AUTOMATICAMENTE": automatico,
                "ACEITA_ID_MANUAL": manual,
                "CONCLUSAO": conclusao,
            }
            relatorio_ids.append(item)
            mensagem("INFO", f"{tabela}.{nome_coluna}: {conclusao}")
except Exception as exc:
    mensagem("ERRO", f"Falha ao avaliar identities: {type(exc).__name__}: {exc}")

mostrar_linhas("IDENTITIES E IDs MANUAIS", relatorio_ids)


## 10. Qualidade dos dados existentes

Para cada tabela, mede quantidade de linhas, nulos em colunas obrigatórias, duplicidade da chave primária e uso de `VARCHAR2`. Para colunas identity, informa mínimo, máximo e quantidade distinta. As consultas são executadas tabela por tabela para isolar falhas.


In [ ]:
%%spark

contagens = []
validacao_nulos = []
validacao_duplicidades = []
perfil_varchar = []
perfil_identities = []

try:
    cabecalho("QUALIDADE DOS DADOS")
    colunas_por_tabela = por_tabela(colunas_meta)
    identidade_por_tabela = por_tabela(identidades)

    for tabela in TABELAS_ENVOLVIDAS:
        try:
            linha = consultar_seguro(
                f"Contagem {tabela}",
                f"SELECT COUNT(*) AS QTD FROM {OWNER}.{tabela}",
                mostrar_sucesso=False,
            )
            qtd = valor_inteiro(linha[0].get("QTD"), None) if linha else None
            contagens.append({"TABELA": tabela, "QTD_REGISTROS": qtd})
            mensagem("OK" if qtd is not None else "AVISO", f"{tabela}: quantidade={qtd}")
        except Exception as exc:
            mensagem("ERRO", f"{tabela}: falha na contagem: {type(exc).__name__}: {exc}")

        try:
            obrigatorias = [
                str(c.get("COLUMN_NAME", "")).upper()
                for c in colunas_por_tabela.get(tabela, [])
                if str(c.get("NULLABLE", "")).upper() == "N"
            ]
            if obrigatorias:
                expressoes = [
                    f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS N_{i}"
                    for i, c in enumerate(obrigatorias)
                ]
                linhas = consultar_seguro(
                    f"Nulos obrigatórios {tabela}",
                    f"SELECT {', '.join(expressoes)} FROM {OWNER}.{tabela}",
                    mostrar_sucesso=False,
                )
                resultado = linhas[0] if linhas else {}
                total_nulos = 0
                detalhes = []
                for i, nome_coluna in enumerate(obrigatorias):
                    qtd_nulos = valor_inteiro(resultado.get(f"N_{i}"), 0)
                    total_nulos += qtd_nulos
                    if qtd_nulos:
                        detalhes.append(f"{nome_coluna}={qtd_nulos}")
                validacao_nulos.append({
                    "TABELA": tabela,
                    "TOTAL_NULOS_INDEVIDOS": total_nulos,
                    "DETALHES": ", ".join(detalhes) if detalhes else "Nenhum",
                })
                mensagem("OK" if total_nulos == 0 else "ERRO", f"{tabela}: nulos obrigatórios={total_nulos}")
        except Exception as exc:
            mensagem("ERRO", f"{tabela}: falha ao avaliar nulos: {type(exc).__name__}: {exc}")

        try:
            pk = CHAVES_PRIMARIAS_ESPERADAS.get(tabela, [])
            if pk:
                grupo = ", ".join(pk)
                linhas = consultar_seguro(
                    f"Duplicidades PK {tabela}",
                    f"""
                    SELECT COUNT(*) AS QTD_GRUPOS_DUPLICADOS
                    FROM (
                        SELECT {grupo}
                        FROM {OWNER}.{tabela}
                        GROUP BY {grupo}
                        HAVING COUNT(*) > 1
                    )
                    """,
                    mostrar_sucesso=False,
                )
                qtd_duplicados = valor_inteiro(linhas[0].get("QTD_GRUPOS_DUPLICADOS"), 0) if linhas else None
                validacao_duplicidades.append({
                    "TABELA": tabela,
                    "CHAVE": grupo,
                    "GRUPOS_DUPLICADOS": qtd_duplicados,
                })
                mensagem(
                    "OK" if qtd_duplicados == 0 else "ERRO",
                    f"{tabela}: grupos duplicados na PK={qtd_duplicados}",
                )
        except Exception as exc:
            mensagem("ERRO", f"{tabela}: falha ao avaliar duplicidades: {type(exc).__name__}: {exc}")

        for coluna_meta in colunas_por_tabela.get(tabela, []):
            try:
                nome_coluna = str(coluna_meta.get("COLUMN_NAME", "")).upper()
                tipo = str(coluna_meta.get("DATA_TYPE", "")).upper()
                if tipo == "VARCHAR2":
                    linhas = consultar_seguro(
                        f"Comprimento {tabela}.{nome_coluna}",
                        f"SELECT MAX(LENGTH({nome_coluna})) AS USADO FROM {OWNER}.{tabela}",
                        mostrar_sucesso=False,
                    )
                    usado = valor_inteiro(linhas[0].get("USADO"), 0) if linhas else None
                    limite = valor_inteiro(coluna_meta.get("CHAR_LENGTH"), None)
                    percentual = round((usado / limite) * 100, 2) if usado is not None and limite else None
                    perfil_varchar.append({
                        "TABELA": tabela,
                        "COLUNA": nome_coluna,
                        "MAX_USADO": usado,
                        "LIMITE": limite,
                        "PERCENTUAL": percentual,
                    })
            except Exception as exc:
                mensagem("ERRO", f"{tabela}.{nome_coluna}: falha no perfil VARCHAR2: {exc}")

        for identidade in identidade_por_tabela.get(tabela, []):
            try:
                nome_coluna = str(identidade.get("COLUMN_NAME", "")).upper()
                linhas = consultar_seguro(
                    f"Perfil identity {tabela}.{nome_coluna}",
                    f"""
                    SELECT MIN({nome_coluna}) AS MENOR_ID,
                           MAX({nome_coluna}) AS MAIOR_ID,
                           COUNT(DISTINCT {nome_coluna}) AS IDS_DISTINTOS,
                           COUNT(*) AS TOTAL
                    FROM {OWNER}.{tabela}
                    """,
                    mostrar_sucesso=False,
                )
                if linhas:
                    perfil_identities.append({
                        "TABELA": tabela,
                        "COLUNA": nome_coluna,
                        **linhas[0],
                    })
            except Exception as exc:
                mensagem("ERRO", f"{tabela}.{nome_coluna}: falha no perfil de identity: {exc}")
except Exception as exc:
    mensagem("ERRO", f"Falha geral na avaliação dos dados: {type(exc).__name__}: {exc}")

mostrar_linhas("CONTAGEM POR TABELA", contagens)
mostrar_linhas("NULOS EM COLUNAS OBRIGATÓRIAS", validacao_nulos)
mostrar_linhas("DUPLICIDADES NAS CHAVES", validacao_duplicidades)
mostrar_linhas("OCUPAÇÃO DE VARCHAR2", perfil_varchar)
mostrar_linhas("FAIXA DOS IDs AUTOMÁTICOS", perfil_identities)


## 11. Chaves estrangeiras, órfãos e ordem segura

Testa registros filhos sem pai, mostra dependências externas e deriva a ordem de carga e de limpeza a partir das FKs reais. Esta etapa merece atenção especial para `PBCO_CADD`, `SGT_CADD`, `RCM_VLDD` e `RCM_VRS`.


In [ ]:
%%spark

validacao_orfaos = []
ordem_carga_calculada = []
ordem_limpeza_calculada = []
ciclos_dependencia = []

try:
    cabecalho("DEPENDÊNCIAS E INTEGRIDADE REFERENCIAL")
    if "Relacionamentos FK" not in estado_avaliacao["consultas_ok"]:
        raise RuntimeError("Metadados de FK indisponíveis; ordem topológica não será inferida.")
    fk_agrupadas = defaultdict(list)
    for item in fks:
        chave = (
            str(item.get("TABLE_NAME", "")).upper(),
            str(item.get("CONSTRAINT_NAME", "")).upper(),
            str(item.get("REFERENCED_OWNER", "")).upper(),
            str(item.get("REFERENCED_TABLE", "")).upper(),
        )
        fk_agrupadas[chave].append(item)

    for (filha, nome_fk, owner_pai, pai), itens in fk_agrupadas.items():
        try:
            itens = sorted(itens, key=lambda x: valor_inteiro(x.get("POSITION"), 9999))
            pares = [
                (str(x.get("COLUMN_NAME", "")).upper(), str(x.get("REFERENCED_COLUMN", "")).upper())
                for x in itens
            ]
            condicao_join = " AND ".join([f"f.{fc} = p.{pc}" for fc, pc in pares])
            filho_preenchido = " AND ".join([f"f.{fc} IS NOT NULL" for fc, _ in pares])
            pai_ausente = " AND ".join([f"p.{pc} IS NULL" for _, pc in pares])

            linhas = consultar_seguro(
                f"Órfãos {nome_fk}",
                f"""
                SELECT COUNT(*) AS QTD_ORFAOS
                FROM {OWNER}.{filha} f
                LEFT JOIN {owner_pai}.{pai} p ON {condicao_join}
                WHERE {filho_preenchido}
                  AND {pai_ausente}
                """,
                mostrar_sucesso=False,
            )
            qtd_orfaos = valor_inteiro(linhas[0].get("QTD_ORFAOS"), None) if linhas else None
            validacao_orfaos.append({
                "TABELA_FILHA": filha,
                "FK": nome_fk,
                "COLUNAS_FILHAS": ", ".join(fc for fc, _ in pares),
                "TABELA_PAI": f"{owner_pai}.{pai}",
                "COLUNAS_PAI": ", ".join(pc for _, pc in pares),
                "QTD_ORFAOS": qtd_orfaos,
            })
            mensagem("OK" if qtd_orfaos == 0 else "ERRO", f"{nome_fk}: órfãos={qtd_orfaos}")
        except Exception as exc:
            mensagem("ERRO", f"{nome_fk}: falha ao testar órfãos: {type(exc).__name__}: {exc}")

    # Topologia: pai deve aparecer antes da filha na carga.
    nos = set(TABELAS_ENVOLVIDAS)
    adjacencia = {t: set() for t in nos}
    grau_entrada = {t: 0 for t in nos}
    for (filha, _, owner_pai, pai), _ in fk_agrupadas.items():
        if owner_pai == OWNER and filha in nos and pai in nos and filha not in adjacencia[pai]:
            adjacencia[pai].add(filha)
            grau_entrada[filha] += 1

    fila = deque([t for t in TABELAS_ENVOLVIDAS if grau_entrada[t] == 0])
    while fila:
        atual = fila.popleft()
        ordem_carga_calculada.append(atual)
        for filha in sorted(adjacencia[atual]):
            grau_entrada[filha] -= 1
            if grau_entrada[filha] == 0:
                fila.append(filha)

    ciclos_dependencia = sorted(nos - set(ordem_carga_calculada))
    ordem_limpeza_calculada = list(reversed(ordem_carga_calculada))

    if ciclos_dependencia:
        mensagem("ERRO", "Ciclo de dependências detectado: " + ", ".join(ciclos_dependencia))
    else:
        mensagem("OK", "Não foram encontrados ciclos entre as tabelas estudadas.")
        mensagem("INFO", "Ordem topológica de carga: " + " -> ".join(ordem_carga_calculada))
        mensagem("INFO", "Ordem topológica de limpeza: " + " -> ".join(ordem_limpeza_calculada))

    externas = [
        x for x in dependencias_externas
        if str(x.get("CHILD_OWNER", "")).upper() != OWNER
        or str(x.get("CHILD_TABLE", "")).upper() not in set(TABELAS_ENVOLVIDAS)
    ]
    if externas:
        mensagem("AVISO", f"Existem {len(externas)} FKs externas que podem impedir limpeza dos pais.")
    else:
        mensagem("OK", "Nenhuma FK externa visível aponta para as tabelas estudadas.")
except Exception as exc:
    mensagem("ERRO", f"Falha na avaliação de dependências: {type(exc).__name__}: {exc}")

mostrar_linhas("VALIDAÇÃO DE REGISTROS ÓRFÃOS", validacao_orfaos)
mostrar_linhas("TODAS AS FKs QUE APONTAM PARA O ESCOPO", dependencias_externas)


## 12. Índices, triggers, comentários, privilégios e estatísticas

Esses itens ajudam a explicar performance, geração indireta de valores, bloqueios de DML, comportamento oculto e qualidade da documentação do schema.


In [ ]:
%%spark

resumo_operacional = []

try:
    indices_por_tabela = por_tabela(indices)
    triggers_por_tabela = por_tabela(triggers)
    privilegios_por_tabela = por_tabela(privilegios)
    comentarios_coluna_por_tabela = por_tabela(comentarios_coluna)
    metas_por_tabela = {str(x.get("TABLE_NAME", "")).upper(): x for x in tabelas_meta}

    cabecalho("ASPECTOS OPERACIONAIS")
    for tabela in TABELAS_ENVOLVIDAS:
        itens_indice = indices_por_tabela.get(tabela, [])
        nomes_indices = sorted({str(x.get("INDEX_NAME", "")) for x in itens_indice if x.get("INDEX_NAME")})
        itens_trigger = triggers_por_tabela.get(tabela, [])
        itens_priv = privilegios_por_tabela.get(tabela, [])
        comentarios_preenchidos = sum(
            1 for x in comentarios_coluna_por_tabela.get(tabela, []) if str(x.get("COMMENTS") or "").strip()
        )
        total_colunas = len(CONTRATO_ESPERADO.get(tabela, {}))
        meta = metas_por_tabela.get(tabela, {})

        triggers_desabilitadas = [x.get("TRIGGER_NAME") for x in itens_trigger if str(x.get("STATUS", "")).upper() != "ENABLED"]
        indices_invalidos = [x.get("INDEX_NAME") for x in itens_indice if str(x.get("STATUS", "")).upper() not in {"VALID", "N/A", ""}]

        resumo_operacional.append({
            "TABELA": tabela,
            "INDICES": len(nomes_indices),
            "INDICES_NAO_VALIDOS": len(set(indices_invalidos)),
            "TRIGGERS": len(itens_trigger),
            "TRIGGERS_DESABILITADAS": len(triggers_desabilitadas),
            "PRIVILEGIOS_VISIVEIS": len(itens_priv),
            "COLUNAS_COM_COMENTARIO": f"{comentarios_preenchidos}/{total_colunas}",
            "NUM_ROWS_ESTIMADO": meta.get("NUM_ROWS"),
            "LAST_ANALYZED": meta.get("LAST_ANALYZED"),
            "PARTITIONED": meta.get("PARTITIONED"),
        })

        if triggers_desabilitadas:
            mensagem("AVISO", f"{tabela}: triggers desabilitadas: {triggers_desabilitadas}")
        if indices_invalidos:
            mensagem("AVISO", f"{tabela}: índices com status de atenção: {sorted(set(indices_invalidos))}")
        if meta.get("LAST_ANALYZED") is None:
            mensagem("AVISO", f"{tabela}: estatísticas sem LAST_ANALYZED visível.")
except Exception as exc:
    mensagem("ERRO", f"Falha no resumo operacional: {type(exc).__name__}: {exc}")

mostrar_linhas("RESUMO OPERACIONAL POR TABELA", resumo_operacional)
mostrar_linhas("ÍNDICES E COLUNAS", indices)
mostrar_linhas("TRIGGERS", triggers)
mostrar_linhas("CONDIÇÕES CHECK", checks)
mostrar_linhas("COMENTÁRIOS DAS TABELAS", comentarios_tabela)
mostrar_linhas("PRIVILÉGIOS", privilegios)


## 13. Resumo executivo e documentação Markdown

Esta célula consolida o que foi encontrado. Ela não grava arquivos: mantém o relatório na variável `documentacao_tabelas_envolvidas_md` e o imprime para revisão/cópia controlada.


In [ ]:
%%spark

documentacao_tabelas_envolvidas_md = ""
resumo_final = []


def md_valor(valor):
    if valor is None or valor == "":
        return "-"
    return str(valor).replace("|", "\\|").replace("\n", " ")


def tabela_markdown(linhas, colunas):
    if not linhas:
        return "_Sem dados disponíveis nesta execução._\n"
    saida = [
        "| " + " | ".join(colunas) + " |",
        "| " + " | ".join(["---"] * len(colunas)) + " |",
    ]
    for linha in linhas:
        saida.append("| " + " | ".join(md_valor(linha.get(c)) for c in colunas) + " |")
    return "\n".join(saida) + "\n"


try:
    schema_status = {x.get("TABELA"): x.get("STATUS") for x in validacao_schema}
    pk_status = {x.get("TABELA"): x.get("STATUS") for x in validacao_pks}
    contagem_status = {x.get("TABELA"): x.get("QTD_REGISTROS") for x in contagens}
    nulos_status = {x.get("TABELA"): x.get("TOTAL_NULOS_INDEVIDOS") for x in validacao_nulos}
    duplic_status = {x.get("TABELA"): x.get("GRUPOS_DUPLICADOS") for x in validacao_duplicidades}
    orfaos_por_tabela = defaultdict(int)
    for x in validacao_orfaos:
        qtd = x.get("QTD_ORFAOS")
        if qtd is not None:
            orfaos_por_tabela[x.get("TABELA_FILHA")] += int(qtd)

    for tabela in TABELAS_ENVOLVIDAS:
        problemas = []
        if schema_status.get(tabela) != "OK":
            problemas.append("schema")
        if pk_status.get(tabela) != "OK":
            problemas.append("PK")
        if (nulos_status.get(tabela) or 0) > 0:
            problemas.append("nulos")
        if (duplic_status.get(tabela) or 0) > 0:
            problemas.append("duplicidades")
        if orfaos_por_tabela.get(tabela, 0) > 0:
            problemas.append("órfãos")

        resumo_final.append({
            "TABELA": tabela,
            "QTD": contagem_status.get(tabela),
            "SCHEMA": schema_status.get(tabela, "NÃO AVALIADO"),
            "PK": pk_status.get(tabela, "NÃO AVALIADO"),
            "NULOS_OBRIGATORIOS": nulos_status.get(tabela),
            "DUPLICIDADES_PK": duplic_status.get(tabela),
            "ORFAOS": orfaos_por_tabela.get(tabela, 0),
            "STATUS_GERAL": "ATENÇÃO: " + ", ".join(problemas) if problemas else "OK",
        })

    agora = datetime.now().isoformat(timespec="seconds")
    partes = [
        "# Documentação das tabelas envolvidas",
        "",
        f"Relatório gerado pelo notebook `avaliacao-oficial-das-tabelas.ipynb` em `{agora}`.",
        "",
        "## Contexto da conexão",
        "",
        f"- Schema avaliado: `{OWNER or 'indisponível'}`",
        f"- Banco: `{contexto_banco.get('DB_NAME', 'indisponível')}`",
        f"- Serviço: `{contexto_banco.get('SERVICE_NAME', 'indisponível')}`",
        "- Operação do notebook: somente leitura",
        "",
        "## Resumo executivo",
        "",
        tabela_markdown(
            resumo_final,
            ["TABELA", "QTD", "SCHEMA", "PK", "NULOS_OBRIGATORIOS", "DUPLICIDADES_PK", "ORFAOS", "STATUS_GERAL"],
        ),
        "## Colunas físicas",
        "",
        tabela_markdown(
            colunas_meta,
            ["TABLE_NAME", "COLUMN_ID", "COLUMN_NAME", "DATA_TYPE", "CHAR_LENGTH", "DATA_PRECISION", "DATA_SCALE", "NULLABLE"],
        ),
        "## Identidades e IDs manuais",
        "",
        tabela_markdown(
            relatorio_ids,
            ["TABELA", "COLUNA", "IDENTITY", "GENERATION_TYPE", "GERADO_AUTOMATICAMENTE", "ACEITA_ID_MANUAL", "CONCLUSAO"],
        ),
        "## Chaves primárias",
        "",
        tabela_markdown(validacao_pks, ["TABELA", "STATUS", "DETALHES"]),
        "## Chaves estrangeiras e órfãos",
        "",
        tabela_markdown(
            validacao_orfaos,
            ["TABELA_FILHA", "FK", "COLUNAS_FILHAS", "TABELA_PAI", "COLUNAS_PAI", "QTD_ORFAOS"],
        ),
        "## Ordem derivada das dependências",
        "",
        "- Carga (pais antes dos filhos): " + (" -> ".join(ordem_carga_calculada) if ordem_carga_calculada else "não calculada"),
        "- Limpeza (filhos antes dos pais): " + (" -> ".join(ordem_limpeza_calculada) if ordem_limpeza_calculada else "não calculada"),
        "- Ciclos: " + (", ".join(ciclos_dependencia) if ciclos_dependencia else "nenhum detectado"),
        "",
        "## Aspectos operacionais",
        "",
        tabela_markdown(
            resumo_operacional,
            ["TABELA", "INDICES", "INDICES_NAO_VALIDOS", "TRIGGERS", "TRIGGERS_DESABILITADAS", "PRIVILEGIOS_VISIVEIS", "COLUNAS_COM_COMENTARIO", "LAST_ANALYZED"],
        ),
        "## Divergências de schema",
        "",
        "\n".join([f"- {x}" for x in divergencias_schema]) if divergencias_schema else "- Nenhuma divergência identificada.",
        "",
        "## Falhas e limitações da avaliação",
        "",
        "\n".join([f"- {x}" for x in estado_avaliacao["erros"]]) if estado_avaliacao["erros"] else "- Nenhuma falha de consulta registrada.",
        "",
        "## Observações importantes",
        "",
        "- `GENERATED ALWAYS` significa que o ID deve ser omitido da carga; informar valor manual explícito causa erro Oracle.",
        "- `AVS_SELD` depende dos catálogos PBCO, SGT, APSC, CNR e TND.",
        "- `RCM_VLDD` depende de `PROJ_CADD`; `RCM_VRS` depende da chave composta de `RCM_VLDD`.",
        "- Estatísticas de `ALL_TABLES.NUM_ROWS` podem estar defasadas; a coluna QTD do resumo vem de `COUNT(*)`.",
        "- Uma consulta ausente no relatório pode indicar falta de privilégio no dicionário; verifique a seção de falhas.",
        "",
    ]
    documentacao_tabelas_envolvidas_md = "\n".join(partes)

    cabecalho("RESUMO FINAL")
    mostrar_linhas("STATUS CONSOLIDADO", resumo_final)
    mensagem("INFO", f"Consultas concluídas: {len(estado_avaliacao['consultas_ok'])}")
    mensagem("INFO", f"Avisos acumulados: {len(estado_avaliacao['avisos'])}")
    mensagem("INFO", f"Erros acumulados: {len(estado_avaliacao['erros'])}")
    mensagem("OK", "Variável documentacao_tabelas_envolvidas_md criada.")
    print("\n" + documentacao_tabelas_envolvidas_md)
except Exception as exc:
    mensagem("ERRO", f"Falha ao consolidar a documentação: {type(exc).__name__}: {exc}")
    traceback.print_exc()


## 14. Encerrar a sessão (opcional)

Execute apenas quando terminar a análise. A célula encerra recursos Spark; não altera as tabelas Oracle.


In [ ]:
try:
    %spark cleanup
    print("[OK] Recursos da sessão Spark encerrados.")
except Exception as exc:
    print(f"[AVISO] Não foi possível encerrar a sessão automaticamente: {type(exc).__name__}: {exc}")
